In [5]:
import sys
from pathlib import Path
import importlib
import src.prepocessing as prep
project_path = Path.cwd().parent
sys.path.insert(0, str(project_path))

from src.prepocessing import preproce, forecast_future
importlib.reload(prep)



<module 'src.prepocessing' from 'f:\\forecasting-challenge\\src\\prepocessing.py'>

## loading the data from preprocessing 

In [6]:
x_train, y_train, y_test, timestamps_test, history = preproce()
print(y_train.dtype)
print(y_train[:3])

float64
[[1.00000000e+04 9.22441916e+00]
 [1.00000000e+04 9.14350337e+00]
 [1.00000000e+04 9.58033480e+00]]


# Testing Deffrent Models:

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import KNeighborsRegressor


models = {
    'RandomForest': RandomForestRegressor(max_depth=100,criterion='friedman_mse',n_estimators=150),

    'DecesionTree': DecisionTreeRegressor(max_depth=100, criterion='squared_error'),

    'GradientBoosting': MultiOutputRegressor(GradientBoostingRegressor(
        loss='squared_error', learning_rate=0.01,
        n_estimators=300,criterion='squared_error',
        max_depth=3,subsample=0.8,random_state=42)
        ),
}


In [12]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

x_train, y_train, y_test, timestamps_test, history = preproce()

# Train and evaluate each model
for model_name, model in models.items():
    print(f"\n{'=' * 15} {model_name} {'=' * 15}")

    #  Train using historical training data
    model.fit(x_train, y_train)

    # forecast 24 future timestamps recursively
    predictions = forecast_future(
        model,
        history,
        steps=len(y_test)
    )

    # 3. Evaluate both targets
    target_names = [
        "Active VM Count",
        "Aggregate CPU Utilization"
    ]

    for i, target_name in enumerate(target_names):
        mae = mean_absolute_error(
            y_test[:, i],
            predictions[:, i]
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test[:, i],
                predictions[:, i]
            )
        )

        print(f"{target_name}:")
        print(f"  MAE:  {mae:.4f}")
        print(f"  RMSE: {rmse:.4f}")


=============== RandomForest ===============
Active VM Count:
  MAE:  0.0000
  RMSE: 0.0000
Aggregate CPU Utilization:
  MAE:  0.1350
  RMSE: 0.1688

=============== DecesionTree ===============
Active VM Count:
  MAE:  0.0000
  RMSE: 0.0000
Aggregate CPU Utilization:
  MAE:  0.1453
  RMSE: 0.1809

=============== GradientBoosting ===============
Active VM Count:
  MAE:  0.0000
  RMSE: 0.0000
Aggregate CPU Utilization:
  MAE:  0.1234
  RMSE: 0.1611


# Predict for both Traget

In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

for model_name, model in models.items():

    print(f"\n-------- {model_name} --------\n")

    model.fit(x_train, y_train)

    pred = forecast_future(
        model,
        history,
        steps=len(y_test)
    )

    print(
        "Active VM Count MAE:",
        mean_absolute_error(y_test[:, 0], pred[:, 0])
    )

    print(
        "Active VM Count RMSE:",
        np.sqrt(mean_squared_error(y_test[:, 0], pred[:, 0]))
    )

    print(
        "CPU Utilization MAE:",
        mean_absolute_error(y_test[:, 1], pred[:, 1])
    )

    print(
        "CPU Utilization RMSE:",
        np.sqrt(mean_squared_error(y_test[:, 1], pred[:, 1]))
    )


-------- RandomForest --------

Active VM Count MAE: 0.0
Active VM Count RMSE: 0.0
CPU Utilization MAE: 0.12813484803603217
CPU Utilization RMSE: 0.1646511193726168

-------- DecesionTree --------

Active VM Count MAE: 0.0
Active VM Count RMSE: 0.0
CPU Utilization MAE: 0.14695047500263417
CPU Utilization RMSE: 0.1847907971630131

-------- GradientBoosting --------

Active VM Count MAE: 0.0
Active VM Count RMSE: 0.0
CPU Utilization MAE: 0.12341768776069843
CPU Utilization RMSE: 0.1610747277240969
